<a href="https://colab.research.google.com/github/DKavya8/chestxray-bias-audit/blob/main/notebooks/06_four_condition_grid.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
!pip install -q torchxrayvision scikit-image

In [3]:
import numpy as np, pandas as pd, os, glob, json, torch
import torchxrayvision as xrv
from sklearn.linear_model import LogisticRegression
print("torch", torch.__version__, "| xrv", xrv.__version__)

torch 2.11.0+cpu | xrv 1.5.4


In [11]:
from google.colab import drive
drive.mount('/content/drive')
BASE = "/content/drive/MyDrive/team-RACK-bias-paper"
RESULTS = f"{BASE}/results"
OUT = f"{RESULTS}/four_condition_grid"; os.makedirs(OUT, exist_ok=True)
print("BASE exists?", os.path.exists(BASE))

Mounted at /content/drive
BASE exists? True


In [8]:
!rm -rf /content/repo && git clone -q https://github.com/DKavya8/chestxray-bias-audit /content/repo
REPO = "/content/repo"
print("splits:", len(glob.glob(f"{REPO}/splits/seed_*")), "seed dirs (expect 10)")
print("thresholds json exists?", os.path.exists(f"{REPO}/results/group_a_densenet/thresholds_by_seed.json"))

splits: 10 seed dirs (expect 10)
thresholds json exists? True


In [15]:
NIH_14 = ["Atelectasis","Consolidation","Infiltration","Pneumothorax","Edema","Emphysema",
          "Fibrosis","Effusion","Pneumonia","Pleural_Thickening","Cardiomegaly","Nodule","Mass","Hernia"]

sc = (scores[['Image Index'] + NIH_14]
      .rename(columns={f: f's_{f}' for f in NIH_14})
      .rename(columns={'Image Index': 'image_index'}))


md = (meta[['image_index','patient_id','age','sex','follow_up'] + NIH_14]
      .rename(columns={f: f'y_{f}' for f in NIH_14}))

img = md.merge(sc, on='image_index', how='inner', validate='one_to_one')
print("merged image-level:", img.shape)

G = (img.sort_values(['patient_id','follow_up'])
        .drop_duplicates('patient_id', keep='first'))


G = G.rename(columns={'patient_id': 'Patient ID'})
G['Patient ID'] = G['Patient ID'].astype(str)
G['age'] = G['age'].astype(float)
G['bin10'] = ((G['age']//10)*10).astype(int).astype(str)

print("G first-scan per patient:", G.shape)
print("sex counts:", dict(G['sex'].value_counts()))
print("bin10:", sorted(G['bin10'].unique(), key=int))
print(G[['Patient ID','sex','age','bin10','y_Atelectasis','s_Atelectasis']].head(3).to_string(index=False))

merged image-level: (112106, 33)
G first-scan per patient: (30797, 34)
sex counts: {'M': np.int64(16625), 'F': np.int64(14172)}
bin10: ['0', '10', '20', '30', '40', '50', '60', '70', '80', '90']
Patient ID sex  age bin10  y_Atelectasis  s_Atelectasis
         1   M 57.0    50              0       0.084476
         2   M 80.0    80              0       0.158287
         3   F 74.0    70              0       0.042101


In [16]:
print("patients in meta:", meta['patient_id'].nunique(), "| in G:", G['Patient ID'].nunique(),
      "| lost:", meta['patient_id'].nunique() - G['Patient ID'].nunique())

patients in meta: 30797 | in G: 30797 | lost: 0


In [17]:
thr_by_seed = json.load(open(f"{REPO}/results/group_a_densenet/thresholds_by_seed.json"))
split_ids = {}
for d in sorted(glob.glob(f"{REPO}/splits/seed_*")):
    seed = d.split('seed_')[1]
    cal  = set(pd.read_csv(f"{d}/calibration_patients.csv")['Patient ID'].astype(str))
    test = set(pd.read_csv(f"{d}/test_patients.csv")['Patient ID'].astype(str))
    split_ids[seed] = (cal, test)

print("seeds:", len(split_ids))
s0 = list(split_ids)[0]
print("example seed", s0, "-> cal", len(split_ids[s0][0]), "test", len(split_ids[s0][1]))

cal0, test0 = split_ids[s0]
print("G∩cal:", G['Patient ID'].isin(cal0).sum(), "| G∩test:", G['Patient ID'].isin(test0).sum())

seeds: 10
example seed 113462462 -> cal 24644 test 6161
G∩cal: 24638 | G∩test: 6159


In [18]:
def pooled_fnr(df, thr, w=None):
    """micro-FNR pooled across the 14 findings, each at its own frozen threshold."""
    w = np.ones(len(df)) if w is None else np.asarray(w, float)
    fn = pos = 0.0
    for f in NIH_14:
        y = df[f'y_{f}'].to_numpy(); s = df[f's_{f}'].to_numpy(); p = (y == 1)
        pos += (w * p).sum()
        fn  += (w * (p & (s < thr[f]))).sum()
    return fn / pos if pos > 0 else np.nan

def sex_gap(df, thr, wcol=None):
    isF = df['sex'].to_numpy() == 'F'
    w = df[wcol].to_numpy() if wcol else np.ones(len(df))
    return pooled_fnr(df[isF], thr, w[isF]) - pooled_fnr(df[~isF], thr, w[~isF])

def exact_bin_match(df, seed):
    """condition 2a: within each bin10, downsample the larger sex to equal counts."""
    rng = np.random.default_rng(seed); keep = []
    for _, g in df.groupby('bin10'):
        F = g[g.sex=='F']; M = g[g.sex=='M']; n = min(len(F), len(M))
        if n == 0: continue
        keep.append(F if len(F)==n else F.sample(n=n, random_state=rng.integers(1<<32)))
        keep.append(M if len(M)==n else M.sample(n=n, random_state=rng.integers(1<<32)))
    return pd.concat(keep) if keep else df.iloc[:0]

def ipw_weights(df):
    """condition 2b: reweight each sex to the pooled (F+M)/2 age-bin distribution."""
    d = df.copy()
    fp = d[d.sex=='F']['bin10'].value_counts(normalize=True)
    mp = d[d.sex=='M']['bin10'].value_counts(normalize=True)
    bins = sorted(set(fp.index)|set(mp.index)); fp = fp.reindex(bins,fill_value=0); mp = mp.reindex(bins,fill_value=0)
    target = (fp+mp)/2
    d['w'] = d.apply(lambda r: target[r.bin10]/((fp if r.sex=='F' else mp)[r.bin10]), axis=1)
    return d

print("functions defined")

functions defined


In [19]:
match_seeds = np.random.default_rng(20260823).integers(0, 1<<32, size=100).tolist()

rows = []
for seed,(cal_ids,test_ids) in split_ids.items():
    test = G[G['Patient ID'].isin(test_ids)]
    thr = thr_by_seed[seed]

    S_orig = sex_gap(test, thr)
    S_match = np.mean([sex_gap(exact_bin_match(test, ms), thr)
                       for ms in match_seeds])
    tw = ipw_weights(test)
    S_ipw = sex_gap(tw, thr, wcol='w')

    rows.append(dict(seed=seed, S_orig=S_orig, S_match=S_match, S_ipw=S_ipw))
    print(f"{seed}:  S_orig={S_orig:+.4f}  S_match={S_match:+.4f}  S_ipw={S_ipw:+.4f}")

cond12 = pd.DataFrame(rows)
print("\nmean across 10 splits:")
print(cond12[['S_orig','S_match','S_ipw']].mean().round(4).to_string())

113462462:  S_orig=+0.0561  S_match=+0.0440  S_ipw=+0.0421
1524358342:  S_orig=+0.0629  S_match=+0.0473  S_ipw=+0.0492
1569714665:  S_orig=+0.0098  S_match=-0.0050  S_ipw=-0.0010
1591287646:  S_orig=-0.0083  S_match=-0.0245  S_ipw=-0.0203
2006902500:  S_orig=+0.0060  S_match=-0.0041  S_ipw=-0.0075
2748406118:  S_orig=+0.0553  S_match=+0.0477  S_ipw=+0.0479
2763601433:  S_orig=+0.0315  S_match=+0.0205  S_ipw=+0.0173
342858866:  S_orig=+0.0067  S_match=-0.0070  S_ipw=-0.0067
3658676649:  S_orig=+0.0208  S_match=+0.0077  S_ipw=+0.0095
768519171:  S_orig=-0.0119  S_match=-0.0257  S_ipw=-0.0225

mean across 10 splits:
S_orig     0.0229
S_match    0.0101
S_ipw      0.0108


In [20]:
def _logit(p):
    p = np.clip(np.asarray(p,float), 1e-6, 1-1e-6); return np.log(p/(1-p))

def fit_calibrators(cal, use_age):
    """per-finding logistic calibrator on cal patients.
       use_age=False -> condition 3 (Platt on logit(score) only).
       use_age=True  -> condition 4 (ASBDC: + standardized age + age^2)."""
    amu, asd = cal['age'].mean(), (cal['age'].std() or 1.0)
    cals, fb = {}, {}
    for f in NIH_14:
        y = cal[f'y_{f}'].to_numpy().astype(int); s = cal[f's_{f}'].to_numpy()
        if y.sum() < 10 or np.unique(y).size < 2:
            cals[f] = None; fb[f] = int(y.sum()); continue
        X = _logit(s).reshape(-1,1)
        if use_age:
            z = (cal['age'].to_numpy()-amu)/asd
            X = np.column_stack([_logit(s), z, z*z])
        cals[f] = (LogisticRegression(C=1e6, max_iter=2000).fit(X, y), amu, asd, use_age)
    return cals, fb

def apply_calibrators(df, cals):
    """return a copy of df with s_{f} replaced by calibrated probabilities."""
    out = df.copy()
    for f in NIH_14:
        c = cals[f]
        if c is None: continue
        clf, amu, asd, use_age = c
        s = df[f's_{f}'].to_numpy()
        X = _logit(s).reshape(-1,1)
        if use_age:
            z = (df['age'].to_numpy()-amu)/asd
            X = np.column_stack([_logit(s), z, z*z])
        out[f's_{f}'] = clf.predict_proba(X)[:,1]
    return out

print("calibrators defined")

calibrators defined


In [21]:
import hashlib
def backbone_fingerprint(model):
    """sha-256 over every weight+buffer, sorted by name — changes iff the backbone changes."""
    h = hashlib.sha256()
    sd = model.state_dict()
    for k in sorted(sd):
        h.update(k.encode()); h.update(sd[k].detach().cpu().contiguous().numpy().tobytes())
    return h.hexdigest()

model = xrv.models.DenseNet(weights="densenet121-res224-all").eval()
fp_before = backbone_fingerprint(model)
print("backbone fingerprint (before):", fp_before[:16], "...\n")

rows, fallbacks = [], {}
for seed,(cal_ids,test_ids) in split_ids.items():
    cal  = G[G['Patient ID'].isin(cal_ids)]
    test = G[G['Patient ID'].isin(test_ids)]
    thr  = thr_by_seed[seed]

    def three(frame):
        so = sex_gap(frame, thr)
        sm = np.mean([sex_gap(exact_bin_match(frame, ms), thr) for ms in match_seeds])
        si = sex_gap(ipw_weights(frame), thr, wcol='w')
        return so, sm, si

    c3, fb3 = fit_calibrators(cal, use_age=False)
    c4, fb4 = fit_calibrators(cal, use_age=True)
    s3o,s3m,s3i = three(apply_calibrators(test, c3))
    s4o,s4m,s4i = three(apply_calibrators(test, c4))
    if fb3: fallbacks[(seed,'c3')] = fb3
    if fb4: fallbacks[(seed,'c4')] = fb4

    rows.append(dict(seed=seed, c3_ipw=s3i, c4_ipw=s4i, c3_match=s3m, c4_match=s4m,
                     dS_ipw=s3i-s4i, dS_match=s3m-s4m))
    print(f"{seed}:  c3_ipw={s3i:+.4f}  c4_ipw={s4i:+.4f}  dS_ipw={s3i-s4i:+.4f}")


fp_after = backbone_fingerprint(model)
assert fp_after == fp_before, "BACKBONE CHANGED — not purely post-hoc!"
print("\nbackbone byte-identical after ASBDC:", fp_after == fp_before)

cond34 = pd.DataFrame(rows)
print("\nmean across 10 splits:")
print(cond34[['c3_ipw','c4_ipw','dS_ipw','c3_match','c4_match','dS_match']].mean().round(4).to_string())
print("\nfallbacks (finding: n_pos in cal, <10 so identity):", fallbacks if fallbacks else "none")

If this fails you can run `wget https://github.com/mlmed/torchxrayvision/releases/download/v1/nih-pc-chex-mimic_ch-google-openi-kaggle-densenet121-d121-tw-lr001-rot45-tr15-sc15-seed0-best.pt -O /root/.torchxrayvision/models_data/nih-pc-chex-mimic_ch-google-openi-kaggle-densenet121-d121-tw-lr001-rot45-tr15-sc15-seed0-best.pt`
[██████████████████████████████████████████████████]
backbone fingerprint (before): c8f0ada55b4c059b ...

113462462:  c3_ipw=+0.0260  c4_ipw=+0.0304  dS_ipw=-0.0045
1524358342:  c3_ipw=+0.0268  c4_ipw=+0.0266  dS_ipw=+0.0002
1569714665:  c3_ipw=-0.0072  c4_ipw=-0.0095  dS_ipw=+0.0023
1591287646:  c3_ipw=-0.0125  c4_ipw=-0.0163  dS_ipw=+0.0038
2006902500:  c3_ipw=-0.0021  c4_ipw=+0.0008  dS_ipw=-0.0029
2748406118:  c3_ipw=+0.0291  c4_ipw=+0.0279  dS_ipw=+0.0012
2763601433:  c3_ipw=+0.0119  c4_ipw=+0.0142  dS_ipw=-0.0024
342858866:  c3_ipw=-0.0024  c4_ipw=+0.0090  dS_ipw=-0.0114
3658676649:  c3_ipw=+0.0044  c4_ipw=+0.0063  dS_ipw=-0.0020
768519171:  c3_ipw=-0.0115  c

In [23]:
cond12.to_csv(f"{OUT}/conditions_1_2_by_split.csv", index=False)
cond34.to_csv(f"{OUT}/conditions_3_4_by_split.csv", index=False)

summary = pd.concat([
    cond12[['S_orig','S_match','S_ipw']].mean(),
    cond34[['c3_ipw','c4_ipw','dS_ipw','c3_match','c4_match','dS_match']].mean()
]).round(4)
summary.to_csv(f"{OUT}/grid_summary_means.csv", header=['mean'])

manifest = {
    "dataset": "NIH ChestX-ray14", "backbone": "densenet121-res224-all",
    "basis": "first-scan-per-patient", "n_patients": int(G['Patient ID'].nunique()),
    "splits": len(split_ids), "match_seeds": 100, "master_seed": 20260823,
    "bin_lock": "bin10 global", "calibrator": "per-finding logistic, C=1e6",
    "condition_3": "age-blind Platt on logit(score)",
    "condition_4": "ASBDC = logit(score)+z_age+z_age^2",
    "backbone_fingerprint": fp_before, "backbone_unchanged": bool(fp_after==fp_before),
    "headline_dS_ipw_mean": float(cond34['dS_ipw'].mean()),
    "headline_dS_match_mean": float(cond34['dS_match'].mean()),
    "fallbacks": {f"{k[0]}_{k[1]}": v for k,v in fallbacks.items()},
}
json.dump(manifest, open(f"{OUT}/run_manifest.json","w"), indent=2)
print("saved to", OUT)
for f in sorted(os.listdir(OUT)): print("  ", f)

saved to /content/drive/MyDrive/team-RACK-bias-paper/results/four_condition_grid
   conditions_1_2_by_split.csv
   conditions_3_4_by_split.csv
   grid_summary_means.csv
   run_manifest.json
